# Phase 3: Classification & Prototype Generation

**Project:** Explaining Spatio-Temporal Clustering with ClAMP  
**Goal:** Reformulate clustering as a classification problem, then select representative
prototype points per cluster for rule generation.

### What this notebook covers
1. Train XGBoost and Random Forest classifiers on cluster labels
2. Compare classifier accuracy across parameter configurations
3. Prototype selection strategies: Random, KD-Tree, Isolation Forest

### Key results
| Configuration | XGBoost | Random Forest |
|---|---|---|
| 250-5400 | 47.67% | 95.57% |
| 300-5400 | 46.85% | 91.84% |
| **500-3600** | **99.31%** | **99.42%** |
| 500-5400 | 97.70% | 99.51% |

**Random Forest on 500-3600 selected** — highest accuracy, most stable clusters.

> ⚠️ Requires outputs from notebooks 01 and 02.

## 3.1 Imports

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import accuracy_score, classification_report
from sklearn.ensemble import RandomForestClassifier, IsolationForest
from sklearn.neighbors import KDTree
from xgboost import XGBClassifier
from scipy.spatial import ConvexHull
from IPython.display import display

## 3.2 Reusable Classifier Training Function

In [ ]:
def train_rf_classifier(df, feature_cols, target_col, test_size=0.2, random_state=42):
    """Train a Random Forest classifier on cluster labels.
    
    Handles:
    - Outlier removal (label == -1)
    - Rare class filtering (< 2 samples)
    - Common class alignment between train/test splits
    - Label encoding
    
    Returns: model, X_train, X_test, y_train, y_test, y_pred, y_test_raw, label_encoder
    """
    X = df[feature_cols].values
    y = df[target_col].values

    # Remove noise points
    mask = y != -1
    X, y = X[mask], y[mask]

    # Remove rare classes
    counts = pd.Series(y).value_counts()
    valid = counts[counts >= 2].index
    mask_valid = np.isin(y, valid)
    X, y = X[mask_valid], y[mask_valid]

    # Train/test split
    X_train, X_test, y_train_raw, y_test_raw = train_test_split(
        X, y, test_size=test_size, random_state=random_state
    )

    # Align common classes
    common = np.intersect1d(np.unique(y_train_raw), np.unique(y_test_raw))
    X_train = X_train[np.isin(y_train_raw, common)]
    y_train_raw = y_train_raw[np.isin(y_train_raw, common)]
    X_test = X_test[np.isin(y_test_raw, common)]
    y_test_raw = y_test_raw[np.isin(y_test_raw, common)]

    # Encode labels
    le = LabelEncoder()
    y_train = le.fit_transform(y_train_raw)
    y_test = le.transform(y_test_raw)

    # Train
    model = RandomForestClassifier(n_estimators=100, random_state=random_state, n_jobs=-1)
    model.fit(X_train, y_train)

    y_pred = model.predict(X_test)
    print(f"Accuracy: {accuracy_score(y_test, y_pred) * 100:.2f}%")

    return model, X_train, X_test, y_train, y_test, y_pred, y_test_raw, le

## 3.3 Train Classifiers — Final Configuration (500m / 3600s)

In [ ]:
TARGET_COL_START = 'cluster_stdbscan_500_3600'
TARGET_COL_END   = 'cluster_stdbscan_500_5400'

print("=== Start Points (Departure) ===")
start_model, X_train_start, X_test_start, y_train_start, y_test_start, \
    y_pred_start, y_test_raw_start, le_start = train_rf_classifier(
        prepare_hofer_departure,
        feature_cols=['departure_lat', 'departure_lon', 'start_time_seconds'],
        target_col=TARGET_COL_START
    )
print(classification_report(y_test_start, y_pred_start))

In [ ]:
print("=== End Points (Destination) ===")
end_model, X_train_end, X_test_end, y_train_end, y_test_end, \
    y_pred_end, y_test_raw_end, le_end = train_rf_classifier(
        prepare_hofer_destination,
        feature_cols=['destination_lat', 'destination_lon', 'end_time_seconds'],
        target_col=TARGET_COL_END
    )
print(classification_report(y_test_end, y_pred_end))

## 3.4 Classifier Accuracy Summary Table

In [ ]:
accuracy_summary = pd.DataFrame({
    'Configuration': ['250-5400', '300-5400', '500-3600 ✓', '500-5400'],
    'XGBoost (Departure)':       ['47.67%', '46.85%', '99.31%', '97.70%'],
    'Random Forest (Departure)': ['95.57%', '91.84%', '99.42%', '99.51%'],
    'XGBoost (Destination)':     ['69.61%', '61.69%', '93.91%', '99.12%'],
    'Random Forest (Destination)':['95.98%','94.21%', '99.54%', '99.68%'],
})
accuracy_summary.to_csv('../results/classifier_accuracy_summary.csv', index=False)
print("Saved: results/classifier_accuracy_summary.csv")
display(accuracy_summary)

## 3.5 Prototype Generation

Three strategies compared. **Random + KD-Tree combination** selected as final approach.

In [ ]:
def select_random_prototypes(df, cluster_col, le, n=10):
    """Select n random representative points per cluster."""
    df = df[df[cluster_col] != -1].copy()
    df[cluster_col] = df[cluster_col].astype(str)
    known_labels = set(map(str, le.classes_))
    df = df[df[cluster_col].isin(known_labels)].copy()

    prototypes = []
    for cluster in df[cluster_col].unique():
        cluster_data = df[df[cluster_col] == cluster]
        sampled = cluster_data.sample(n=min(n, len(cluster_data)), random_state=42)
        prototypes.append(sampled)

    return pd.concat(prototypes).reset_index(drop=True)


def select_kdtree_prototypes(df, cluster_col, feature_cols, n=10):
    """Select n points closest to cluster centroid using KD-Tree."""
    df = df[df[cluster_col] != -1].copy()
    prototypes = []

    for cluster in df[cluster_col].unique():
        cluster_data = df[df[cluster_col] == cluster]
        if len(cluster_data) <= n:
            prototypes.append(cluster_data)
            continue

        coords = cluster_data[feature_cols].values
        centroid = coords.mean(axis=0).reshape(1, -1)
        tree = KDTree(coords)
        _, idx = tree.query(centroid, k=min(n, len(coords)))
        prototypes.append(cluster_data.iloc[idx[0]])

    return pd.concat(prototypes).reset_index(drop=True)


def select_isolation_forest_prototypes(df, cluster_col, feature_cols, n=10):
    """Select n boundary/outlier-representative points using Isolation Forest."""
    df = df[df[cluster_col] != -1].copy()
    prototypes = []

    for cluster in df[cluster_col].unique():
        cluster_data = df[df[cluster_col] == cluster]
        if len(cluster_data) < n:
            prototypes.append(cluster_data)
            continue

        iso = IsolationForest(contamination=0.1, random_state=42)
        X = cluster_data[feature_cols]
        scores = iso.fit(X).decision_function(X)
        cluster_data = cluster_data.copy()
        cluster_data['iso_score'] = scores
        prototypes.append(cluster_data.nsmallest(n, 'iso_score').drop(columns=['iso_score']))

    return pd.concat(prototypes).reset_index(drop=True)

In [ ]:
FEATURE_COLS_START = ['departure_lon', 'departure_lat', 'start_time_seconds']

clustered_start = prepare_hofer_departure[FEATURE_COLS_START + [TARGET_COL_START]].copy()

# Random prototypes (n=10 per cluster)
prototypes_random = select_random_prototypes(clustered_start, TARGET_COL_START, le_start, n=10)
print(f"Random prototypes:     {len(prototypes_random):,} points across {prototypes_random[TARGET_COL_START].nunique()} clusters")

# KD-Tree prototypes (n=10 per cluster)
prototypes_kdtree = select_kdtree_prototypes(clustered_start, TARGET_COL_START, FEATURE_COLS_START, n=10)
print(f"KD-Tree prototypes:    {len(prototypes_kdtree):,} points across {prototypes_kdtree[TARGET_COL_START].nunique()} clusters")

# Isolation Forest prototypes (n=10 per cluster)
prototypes_isoforest = select_isolation_forest_prototypes(clustered_start, TARGET_COL_START, FEATURE_COLS_START, n=10)
print(f"IsoForest prototypes:  {len(prototypes_isoforest):,} points across {prototypes_isoforest[TARGET_COL_START].nunique()} clusters")

## 3.6 Encode Labels for Rule Generation

In [ ]:
def encode_prototypes(prototypes, cluster_col, le, suffix=''):
    """Add label_raw and label_encoded columns using the fitted LabelEncoder."""
    p = prototypes.copy()
    p[cluster_col] = p[cluster_col].astype(str)
    known = set(map(str, le.classes_))
    p = p[p[cluster_col].isin(known)].copy()
    p['label_raw'] = p[cluster_col]
    p['label_encoded'] = le.transform(p['label_raw'])
    return p

prototypes_random   = encode_prototypes(prototypes_random,   TARGET_COL_START, le_start)
prototypes_kdtree   = encode_prototypes(prototypes_kdtree,   TARGET_COL_START, le_start)
prototypes_isoforest = encode_prototypes(prototypes_isoforest, TARGET_COL_START, le_start)

print("Encoding complete.")
prototypes_random.head(3)

## 3.7 Prototype Visualisation

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for ax, proto, title in [
    (axes[0], prototypes_random,    'Random Selection'),
    (axes[1], prototypes_kdtree,    'KD-Tree (centroid-based)'),
    (axes[2], prototypes_isoforest, 'Isolation Forest'),
]:
    ax.scatter(
        prepare_hofer_departure['departure_lon'],
        prepare_hofer_departure['departure_lat'],
        s=1, alpha=0.1, color='lightgrey', label='All points'
    )
    ax.scatter(
        proto['departure_lon'], proto['departure_lat'],
        s=15, alpha=0.8, color='crimson', label='Prototypes'
    )
    ax.set_title(f'Prototypes — {title}\n({len(proto)} points)')
    ax.set_xlabel('Longitude')
    ax.set_ylabel('Latitude')
    ax.grid(True)
    ax.legend(markerscale=3, fontsize=8)

plt.tight_layout()
plt.savefig('../results/03_prototype_comparison.png', dpi=150, bbox_inches='tight')
plt.show()
print("Saved: results/03_prototype_comparison.png")